SoundWall converter. This python script will grab data from the spreadsheets datavv.xlsx, datacc.xlsx, datavv2.xlsx and datacc2.xlsx and produce a json file that will construct the soundwall interactive.

In [62]:
import pandas as pd

def excel_to_json_string(input_file, output_file):
    # Read the Excel file
    excel_file = pd.ExcelFile(input_file)
    all_data = []

    # Iterate through each sheet in the Excel file
    for sheet_index, sheet_name in enumerate(excel_file.sheet_names):
        df = pd.read_excel(excel_file, sheet_name=sheet_name)
        sheet_suffix = f"{sheet_index + 1:02d}"  # Ensure unique IDs for each tab

        # Advanced Constructor
        if input_file in ['datacc2.xlsx', 'datavv2.xlsx']:
            first_row = df.iloc[0]
            vlink = str(first_row.get("vlink", ""))
            all_data.append(
                "{\n"
                '  "type": "icn",\n'
                '  "role": "button",\n'
                '  "alt": "' + first_row.get("alt", "").replace('"', '\\"') + '",\n'
                '  "id": "icn' + sheet_suffix + '",\n'
                '  "content": "' + first_row.get("content", "").replace('"', '\\"') + '",\n'
                '  "left": "' + first_row.get("left", "").replace('"', '\\"') + '",\n'
                '  "top": "' + first_row.get("top", "").replace('"', '\\"') + '",\n'
                '  "vlink": "' + vlink + '",\n'
                '  "action": "openGroupGhost",\n'
                '  "target": "grp' + sheet_suffix + '"\n'
                '}'
            )

            group = (
                "{\n"
                '  "type": "grp",\n'
                '  "id": "grp' + sheet_suffix + '",\n'
                '  "style": "grpAdv",\n'
                '  "left": "4em",\n'
                '  "top": "2em",\n'
                '  "width": "56em",\n'
                '  "height": "25em",\n'
                '  "visible": "false",\n'
                '  "children": [\n'
            )

            children_data = []
            for index, row in df.iterrows():
                if index == 0:
                    continue  # Skip the first row
                child_str = '    {\n'
                for key, value in row.items():
                    if pd.notna(value) and value != "":
                        child_str += '      "' + key + '": "' + str(value).replace('"', '\\"') + '",\n'
                child_str = child_str.rstrip(",\n") + "\n    }"  # Remove trailing comma
                if child_str.strip() != "{\n    }":  # Ensure that only valid children are added
                    children_data.append(child_str)
            
            if children_data:  # Add children only if there are valid entries
                group += ',\n'.join(children_data) + '\n'
            group += '  ]\n'
            group += '}\n'

            all_data.append(group)

        # Basic Constructor
        elif input_file in ['datacc.xlsx', 'datavv.xlsx']:
            row1 = df.iloc[0] if not df.empty else pd.Series()  # Handle possible empty DataFrame
            row2 = df.iloc[1] if len(df) > 1 else pd.Series()
            row3 = df.iloc[2] if len(df) > 2 else pd.Series()
            
            displayText = row3.get("text", "default_text") if row3.get("style", "") != 'soundwallSub vector' else 'vector'

            icn_str = (
                '{\n'
                '  "type": "icn",\n'
                '  "role": "image",\n'
                '  "alt": "' + row1.get("alt", "").replace('"', '\\"') + '",\n'
                '  "id": "' + row1.get("alt", "").replace('"', '\\"') + '",\n'
                '  "content": "' + row1.get("content", "").replace('"', '\\"') + '",\n'
                '  "left": "' + row1.get("left", "").replace('"', '\\"') + '",\n'
                '  "top": "' + row1.get("top", "").replace('"', '\\"') + '",\n'
                '  "action": "toggleMulti",\n'
                '  "target": "grp' + sheet_suffix + '",\n'
                '  "children": [\n'
            )
            
            btn_child = (
                '    {\n'
                '      "type": "btn",\n'
                '      "id": "' + str(row2.get('id', 'default_id')) + '",\n'
                '      "alt": "' + str(row2.get('alt', 'default_id')) + '",\n'
                '      "name": "' + str(row2.get('alt', 'default_id')) + '",\n'
                '      "style": "hotspot",\n'
                '      "content": "",\n'
                '      "left": "0",\n'
                '      "top": "0em",\n'
                '      "height": "9.5em",\n'
                '      "width": "9.5em"\n'
                '    }'
            )
            
            grp_child = (
                '    {\n'
                '      "type": "grp",\n'
                '      "id": "' + str(row3.get('id', 'default_id')) + '",\n'
                '      "content": "' + str(row3.get('content', 'default_content')) + '",\n'
                '      "alt": "' + str(row3.get('alt', 'default_id')) + '",\n'
                '      "left": ".4em",\n'
                '      "top": "11em",\n'
                '      "width": "9em",\n'
                '      "height": "12em",\n'
                '      "visible": "false",\n'
                '      "children": [\n'
            )
            
            txt_child = (
                '        {\n'
                '          "type": "txt",\n'
                '          "content": "' + displayText + '",\n'
                '          "left": "-.15em",\n'
                '          "top": "-.15em"\n'
                '        }'
            )
            
            img_child = (
                '        {\n'
                '          "type": "img",\n'
                '          "style": "photo",\n'
                '          "alt": "' + str(row3.get('alt', 'default_alt')) + '",\n'
                '          "content": "' + str(row3.get('content', 'default_content')) + '",\n'
                '          "left": ".5em",\n'
                '          "top": "3.5em",\n'
                '          "width": "7.5em",\n'
                '          "height": "auto"\n'
                '        }'
            )

            grp_child += txt_child + ',\n' + img_child + '\n'
            grp_child += '      ]\n'
            grp_child += '    }\n'
            
            icn_str += btn_child + ',\n' + grp_child
            icn_str += '  ]\n}\n'
            all_data.append(icn_str)

    # Write to JSON file with utf-8 encoding
    with open(output_file, 'w', encoding='utf-8') as json_file:
        json_file.write(',\n'.join(all_data))

# Specify the input and output file paths for each Excel file
file_mappings = {
    "datavv.xlsx": "output1.json",
    "datavv2.xlsx": "output2.json",
    "datacc.xlsx": "output3.json",
    "datacc2.xlsx": "output4.json"
}

# Process each file
for input_file, output_file in file_mappings.items():
    excel_to_json_string(input_file, output_file)


Now we will insert the json snippets into the framework file.

In [63]:
import os

def replace_placeholder_with_json(shell_file, output_files, final_output):
    # Read the shell file content
    with open(shell_file, 'r', encoding='utf-8') as shell:
        shell_content = shell.read()

    # Replace placeholders with the corresponding output file content
    for output_file in output_files:
        placeholder = f'/*ADD {os.path.basename(output_file)} here*/'
        with open(output_file, 'r', encoding='utf-8') as output:
            output_content = output.read()
        shell_content = shell_content.replace(placeholder, output_content)

    # Write the final content to the new file
    with open(final_output, 'w', encoding='utf-8') as final:
        final.write(shell_content)

# Paths to the files
shell_file = '../data/soundwallShell.json'
output_files = [
    'output1.json',
    'output2.json',
    'output3.json',
    'output4.json'
]
final_output = '../data/soundwall.json'

# Perform the replacement
replace_placeholder_with_json(shell_file, output_files, final_output)

print(f'Final JSON file created at {final_output}')


Final JSON file created at ../data/soundwall.json
